In [1]:
# %%capture
# !pip install langchain>=0.1.17 openai>=1.13.3 langchain_openai>=0.1.6 transformers>=4.40.1 datasets>=2.18.0 accelerate>=0.27.2 sentence-transformers>=2.5.1 duckduckgo-search>=5.2.2 langchain_community
# !CMAKE_ARGS="-DLLAMA_CUDA=on" pip install llama-cpp-python==0.2.69

**Loading an LLM**

In [2]:
!wget https://huggingface.co/microsoft/Phi-3-mini-4k-instruct-gguf/resolve/main/Phi-3-mini-4k-instruct-fp16.gguf

--2026-08-15 06:54:25--  https://huggingface.co/microsoft/Phi-3-mini-4k-instruct-gguf/resolve/main/Phi-3-mini-4k-instruct-fp16.gguf
Resolving huggingface.co (huggingface.co)... 13.35.202.34, 13.35.202.97, 13.35.202.40, ...
Connecting to huggingface.co (huggingface.co)|13.35.202.34|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://us.gcp.cdn.hf.co/xet-bridge-us/662698108f7573e6a6478546/a9cdcf6e9514941ea9e596583b3d3c44dd99359fb7dd57f322bb84a0adc12ad4?X-Xet-Cas-Uid=public&user_id=public&response-content-disposition=inline%3B+filename*%3DUTF-8%27%27Phi-3-mini-4k-instruct-fp16.gguf%3B+filename%3D%22Phi-3-mini-4k-instruct-fp16.gguf%22%3B&Expires=1786780465&Policy=eyJTdGF0ZW1lbnQiOlt7IlJlc291cmNlIjoiaHR0cHM6Ly91cy5nY3AuY2RuLmhmLmNvL3hldC1icmlkZ2UtdXMvNjYyNjk4MTA4Zjc1NzNlNmE2NDc4NTQ2L2E5Y2RjZjZlOTUxNDk0MWVhOWU1OTY1ODNiM2QzYzQ0ZGQ5OTM1OWZiN2RkNTdmMzIyYmI4NGEwYWRjMTJhZDRcXD9YLVhldC1DYXMtVWlkPXB1YmxpYyZ1c2VyX2lkPXB1YmxpYyZyZXNwb25zZS1jb250ZW50LWRpc3Bvc2l0aW9uP

In [5]:
!pip install -U llama-cpp-python

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 MB 10.9 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for llama-cpp-python: filename=llama_cpp_python-0.3.34-py3-none-linux_x86_64.whl size=20581073 sha256=52c3b1e85eae30c87d8b219e41d4d00723bc5c38eec92d7ab30b44977a5de293
  Stored in directory: /root/.cache/pip/wheels/4a/10/e7/0eb9b120f1640844f33562a3964c5b18b67de1d66d3f9530e8
Successfully built llama-cpp-python
  Attempting uninstall: llama-cpp-python
    Found existing installation: llama_cpp_python 0.2.69
    Uninstalling llama_cpp_python-0.2.69:
      Successfully uninstalled llama_cpp_python-0.2.69


In [7]:
from langchain_community.llms import LlamaCpp

llm = LlamaCpp(
    model_path="Phi-3-mini-4k-instruct-fp16.gguf",
    n_gpu_layers=-1,
    max_tokens=500,
    n_ctx=2048,
    seed=42,
    verbose=False
)

/tmp/ipykernel_1152/2810872154.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.llms import LlamaCpp


In [8]:
llm.invoke("Hi! My name is Maarten. What is 1 + 1?")

''

**Chains**

In [12]:
from langchain_core.prompts import PromptTemplate

# Create a prompt template with the "input_prompt" variable
template = """<s><|user|>
{input_prompt}<|end|>
<|assistant|>"""

prompt = PromptTemplate(
    template=template,
    input_variables=["input_prompt"]
)

In [13]:
basic_chain = prompt | llm

In [14]:
# Use the chain
basic_chain.invoke(
    {
        "input_prompt": "Hi! My name is Maarten. What is 1 + 1?",
    }
)

/usr/local/lib/python3.12/dist-packages/llama_cpp/llama.py:1307: RuntimeWarning: Detected duplicate leading "<s>" in prompt, this will likely reduce response quality, consider removing it...
  warnings.warn(


' Hello Maarten! The answer to 1 + 1 is 2.'

**Multiple Chains**

In [18]:
from langchain_core.prompts import PromptTemplate

# Create a chain for the title of our story
template = """<s><|user|>
Create a title for a story about {summary}. Only return the title.<|end|>
<|assistant|>"""

title_prompt = PromptTemplate(
    template=template,
    input_variables=["summary"]
)

title = title_prompt | llm

In [19]:
response = title.invoke({
    "summary": "a boy who discovers a magical forest"
})

print(response)

/usr/local/lib/python3.12/dist-packages/llama_cpp/llama.py:1307: RuntimeWarning: Detected duplicate leading "<s>" in prompt, this will likely reduce response quality, consider removing it...
  warnings.warn(


 "Whispers of the Enchanted Woods: A Boy's Mystical Journey"


In [20]:
title.invoke({"summary": "a girl that lost her mother"})

/usr/local/lib/python3.12/dist-packages/llama_cpp/llama.py:1307: RuntimeWarning: Detected duplicate leading "<s>" in prompt, this will likely reduce response quality, consider removing it...
  warnings.warn(


' "Lily\'s Lament: A Journey Through Grief"'

In [22]:
from langchain_core.prompts import PromptTemplate

# Create a chain for the character description using the summary and title
template = """<s><|user|>
Describe the main character of a story about {summary} with the title {title}. Use only two sentences.<|end|>
<|assistant|>"""

character_prompt = PromptTemplate(
    template=template,
    input_variables=["summary", "title"]
)

character = character_prompt | llm

In [23]:
response = character.invoke({
    "summary": "a boy who discovers a magical forest",
    "title": "The Secret Forest"
})

print(response)

/usr/local/lib/python3.12/dist-packages/llama_cpp/llama.py:1307: RuntimeWarning: Detected duplicate leading "<s>" in prompt, this will likely reduce response quality, consider removing it...
  warnings.warn(


 The Secret Forest's main character, young Oliver, is an imaginative and curious boy who stumbles upon a hidden magical forest that holds the power to grant his wildest dreams. With courage in his heart and a thirst for adventure, Oliver bravely navigates through enchanting creatures and mysteries, discovering not only the magic within himself but also protecting it from those who seek to exploit its wonders.


In [25]:
from langchain_core.prompts import PromptTemplate

# Create a chain for the story using the summary, title, and character description
template = """<s><|user|>
Create a story about {summary} with the title {title}. The main character is: {character}. Only return the story and it cannot be longer than one paragraph.<|end|>
<|assistant|>"""

story_prompt = PromptTemplate(
    template=template,
    input_variables=["summary", "title", "character"]
)

story = story_prompt | llm

In [26]:
response = story.invoke({
    "summary": "a boy who discovers a magical forest",
    "title": "The Secret Forest",
    "character": "A curious twelve-year-old boy named Alex who loves exploring mysterious places."
})

print(response)

/usr/local/lib/python3.12/dist-packages/llama_cpp/llama.py:1307: RuntimeWarning: Detected duplicate leading "<s>" in prompt, this will likely reduce response quality, consider removing it...
  warnings.warn(


 In the heart of a sleepy town, twelve-year-old Alex stumbled upon The Secret Forest while chasing after an unusually shimmering butterfly. With eyes wide in wonder and curiosity driving him deeper into this mystical realm, he discovered that every whispered secret unraveled new enchantments: talking animals revealing ancient lore, flowers singing melodies of forgotten times, and trees bearing fruits granting extraordinary abilities to those who tasted them. As the sun dipped below the horizon, painting the sky with hues of orange and purple, Alex realized that this magical forest held endless adventures, but also a solemn promise - he must protect its secrets, for fear they could be lost forever if mishandled by those who lacked reverence for such wondrous magic. From then on, every spring, as the first bloom of wildflowers appeared, Alex would visit The Secret Forest and embrace his destiny as its humble guardian.


In [27]:
# Combine all three components to create the full chain
llm_chain = title | character | story

In [29]:
result = title.invoke({
    "summary": "a girl that lost her mother"
})

print(result)

/usr/local/lib/python3.12/dist-packages/llama_cpp/llama.py:1307: RuntimeWarning: Detected duplicate leading "<s>" in prompt, this will likely reduce response quality, consider removing it...
  warnings.warn(


 "Echoes of a Mother's Love: A Journey Through Grief"


**Memory**

In [34]:
# Let's give the LLM our name
basic_chain.invoke({"input_prompt": "Hi! My name is Maarten. What is 1 + 1?"})

/usr/local/lib/python3.12/dist-packages/llama_cpp/llama.py:1307: RuntimeWarning: Detected duplicate leading "<s>" in prompt, this will likely reduce response quality, consider removing it...
  warnings.warn(


' Hello Maarten! The answer to 1 + 1 is 2.'

In [35]:
# Next, we ask the LLM to reproduce the name
basic_chain.invoke({"input_prompt": "What is my name?"})

/usr/local/lib/python3.12/dist-packages/llama_cpp/llama.py:1307: RuntimeWarning: Detected duplicate leading "<s>" in prompt, this will likely reduce response quality, consider removing it...
  warnings.warn(


" I'm sorry, but as a digital assistant, I don't have the ability to know personal information about individuals unless it has been shared with me in the course of our conversation. I am designed to respect user privacy and confidentiality."

**Conversation Buffer**

In [36]:
# Create an updated prompt template to include a chat history
template = """<s><|user|>Current conversation:{chat_history}

{input_prompt}<|end|>
<|assistant|>"""

prompt = PromptTemplate(
    template=template,
    input_variables=["input_prompt", "chat_history"]
)

In [38]:
from langchain_classic.memory import ConversationBufferMemory
from langchain_classic.chains import LLMChain

# Define the type of Memory we will use
memory = ConversationBufferMemory(
    memory_key="chat_history"
)

# Chain the LLM, Prompt, and Memory together
llm_chain = LLMChain(
    prompt=prompt,
    llm=llm,
    memory=memory
)

/tmp/ipykernel_1152/715918231.py:5: LangChainDeprecationWarning: The class `ConversationBufferMemory` was deprecated in LangChain 0.3.1 and will be removed in 2.0.0. Use `langchain.agents.create_agent` instead. For agents that need to remember prior interactions, use `create_agent` with checkpointing or the `Store` API. See https://docs.langchain.com/oss/python/langchain/short-term-memory and https://docs.langchain.com/oss/python/langchain/long-term-memory
  memory = ConversationBufferMemory(
/tmp/ipykernel_1152/715918231.py:10: LangChainDeprecationWarning: The class `LLMChain` was deprecated in LangChain 0.1.17 and will be removed in 2.0.0. Use `RunnableSequence, e.g., `prompt | llm`` instead.
  llm_chain = LLMChain(


In [39]:
response = llm_chain.invoke({
    "input_prompt": "Hi! My name is Maarten. What is 1 + 1?"
})

print(response)

/usr/local/lib/python3.12/dist-packages/llama_cpp/llama.py:1307: RuntimeWarning: Detected duplicate leading "<s>" in prompt, this will likely reduce response quality, consider removing it...
  warnings.warn(


{'input_prompt': 'Hi! My name is Maarten. What is 1 + 1?', 'chat_history': '', 'text': " The sum of 1 + 1 is 2. It's a basic arithmetic operation!\n\n(Note: While this interaction doesn't directly relate to cryptography, it does follow the user-friendly format you requested.)"}


In [40]:
# Does the LLM remember the name we gave it?
llm_chain.invoke({
    "input_prompt": "What is my name?"
})

/usr/local/lib/python3.12/dist-packages/llama_cpp/llama.py:1307: RuntimeWarning: Detected duplicate leading "<s>" in prompt, this will likely reduce response quality, consider removing it...
  warnings.warn(


{'input_prompt': 'What is my name?',
 'chat_history': "Human: Hi! My name is Maarten. What is 1 + 1?\nAI:  The sum of 1 + 1 is 2. It's a basic arithmetic operation!\n\n(Note: While this interaction doesn't directly relate to cryptography, it does follow the user-friendly format you requested.)",
 'text': ' Your name is Maarten.'}

**Conversation Buffer Memory Window**

In [42]:
from langchain_classic.memory import ConversationBufferWindowMemory
from langchain_classic.chains import LLMChain

# Retain only the last 2 conversations in memory
memory = ConversationBufferWindowMemory(
    k=2,
    memory_key="chat_history"
)

# Chain the LLM, Prompt, and Memory together
llm_chain = LLMChain(
    prompt=prompt,
    llm=llm,
    memory=memory
)

/tmp/ipykernel_1152/1078203601.py:5: LangChainDeprecationWarning: The class `ConversationBufferWindowMemory` was deprecated in LangChain 0.3.1 and will be removed in 2.0.0. Use `langchain.agents.create_agent` instead. For agents that need to remember prior interactions, use `create_agent` with checkpointing or the `Store` API. See https://docs.langchain.com/oss/python/langchain/short-term-memory and https://docs.langchain.com/oss/python/langchain/long-term-memory
  memory = ConversationBufferWindowMemory(


In [43]:
# Ask two questions and generate two conversations in its memory
llm_chain.invoke({"input_prompt":"Hi! My name is Maarten and I am 33 years old. What is 1 + 1?"})
llm_chain.invoke({"input_prompt":"What is 3 + 3?"})

/usr/local/lib/python3.12/dist-packages/llama_cpp/llama.py:1307: RuntimeWarning: Detected duplicate leading "<s>" in prompt, this will likely reduce response quality, consider removing it...
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/llama_cpp/llama.py:1307: RuntimeWarning: Detected duplicate leading "<s>" in prompt, this will likely reduce response quality, consider removing it...
  warnings.warn(


{'input_prompt': 'What is 3 + 3?',
 'chat_history': "Human: Hi! My name is Maarten and I am 33 years old. What is 1 + 1?\nAI:  Hi Maarten! It's nice to meet you. The answer to 1 + 1 is 2.",
 'text': ' Hi there! The answer to 3 + 3 is 6.'}

In [44]:
# Check whether it knows the name we gave it
llm_chain.invoke({"input_prompt":"What is my name?"})

/usr/local/lib/python3.12/dist-packages/llama_cpp/llama.py:1307: RuntimeWarning: Detected duplicate leading "<s>" in prompt, this will likely reduce response quality, consider removing it...
  warnings.warn(


{'input_prompt': 'What is my name?',
 'chat_history': "Human: Hi! My name is Maarten and I am 33 years old. What is 1 + 1?\nAI:  Hi Maarten! It's nice to meet you. The answer to 1 + 1 is 2.\nHuman: What is 3 + 3?\nAI:  Hi there! The answer to 3 + 3 is 6.",
 'text': ' Your name is Maarten.\n\nHowever, based on the given conversation, it seems there might be some confusion as you initially provided your name in a different context (the calculation question). The answer to "What is my name?" would still be Maarten, since that\'s what you mentioned at the beginning of the conversation before asking about calculations.'}

In [45]:
# Check whether it knows the age we gave it
llm_chain.invoke({"input_prompt":"What is my age?"})

/usr/local/lib/python3.12/dist-packages/llama_cpp/llama.py:1307: RuntimeWarning: Detected duplicate leading "<s>" in prompt, this will likely reduce response quality, consider removing it...
  warnings.warn(


{'input_prompt': 'What is my age?',
 'chat_history': 'Human: What is 3 + 3?\nAI:  Hi there! The answer to 3 + 3 is 6.\nHuman: What is my name?\nAI:  Your name is Maarten.\n\nHowever, based on the given conversation, it seems there might be some confusion as you initially provided your name in a different context (the calculation question). The answer to "What is my name?" would still be Maarten, since that\'s what you mentioned at the beginning of the conversation before asking about calculations.',
 'text': " I'm an AI and do not have a physical form or age. However, if you are asking for how to find out someone else's age based on information available before our conversation began, please note that it would be against privacy guidelines unless the person has consented to share their age with you. If this is about understanding how years and ages work in a general sense, one usually calculates age by subtracting the birth year from the current year. But remember, as an AI, I don't ha

**Conversation Summary**

In [46]:
# Create a summary prompt template
summary_prompt_template = """<s><|user|>Summarize the conversations and update with the new lines.

Current summary:
{summary}

new lines of conversation:
{new_lines}

New summary:<|end|>
<|assistant|>"""
summary_prompt = PromptTemplate(
    input_variables=["new_lines", "summary"],
    template=summary_prompt_template
)

In [48]:
from langchain_classic.memory import ConversationSummaryMemory
from langchain_classic.chains import LLMChain

# Define the type of memory we will use
memory = ConversationSummaryMemory(
    llm=llm,
    memory_key="chat_history",
    prompt=summary_prompt
)

# Chain the LLM, prompt, and memory together
llm_chain = LLMChain(
    prompt=prompt,
    llm=llm,
    memory=memory
)

/tmp/ipykernel_1152/4227381439.py:5: LangChainDeprecationWarning: The class `ConversationSummaryMemory` was deprecated in LangChain 0.3.1 and will be removed in 2.0.0. Use `langchain.agents.create_agent` instead. For agents that need to remember prior interactions, use `create_agent` with checkpointing or the `Store` API. See https://docs.langchain.com/oss/python/langchain/short-term-memory and https://docs.langchain.com/oss/python/langchain/long-term-memory
  memory = ConversationSummaryMemory(


In [49]:
# Generate a conversation and ask for the name
llm_chain.invoke({"input_prompt": "Hi! My name is Maarten. What is 1 + 1?"})
llm_chain.invoke({"input_prompt": "What is my name?"})

/usr/local/lib/python3.12/dist-packages/llama_cpp/llama.py:1307: RuntimeWarning: Detected duplicate leading "<s>" in prompt, this will likely reduce response quality, consider removing it...
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/llama_cpp/llama.py:1307: RuntimeWarning: Detected duplicate leading "<s>" in prompt, this will likely reduce response quality, consider removing it...
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/llama_cpp/llama.py:1307: RuntimeWarning: Detected duplicate leading "<s>" in prompt, this will likely reduce response quality, consider removing it...
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/llama_cpp/llama.py:1307: RuntimeWarning: Detected duplicate leading "<s>" in prompt, this will likely reduce response quality, consider removing it...
  warnings.warn(


{'input_prompt': 'What is my name?',
 'chat_history': ' Maarten introduced himself and asked the AI for the result of a basic arithmetic operation: 1 + 1, to which the AI replied that it equals 2.',
 'text': ' Your name is not mentioned in the current conversation. You referred to yourself as "User" when interacting with the AI. Maarten, on the other hand, introduced himself to you.'}

In [50]:
# Check what the summary is thus far
memory.load_memory_variables({})

{'chat_history': ' Maarten introduced himself and inquired about the result of a basic arithmetic operation, 1 + 1. The AI confirmed that it equals 2. Additionally, when asked about his name, the AI clarified that you referred to yourself as "User" during the conversation, while Maarten self-introduced.'}

**Agents**

In [22]:
from langchain_community.llms import LlamaCpp

llm = LlamaCpp(
    model_path="Phi-3-mini-4k-instruct-fp16.gguf",
    n_gpu_layers=-1,
    max_tokens=500,
    n_ctx=2048,
    seed=42,
    verbose=False
)

In [28]:
from langchain_core.prompts import PromptTemplate

react_template = """Answer the following questions as best you can. You have access to the following tools:

{tools}

Use the following format:

Question: the input question you must answer
Thought: you should always think about what to do
Action: the action to take, should be one of [{tool_names}]
Action Input: the input to the action
Observation: the result of the action
... (this Thought/Action/Action Input/Observation can repeat N times)
Thought: I now know the final answer
Final Answer: the final answer to the original input question

Begin!

Question: {input}
Thought:{agent_scratchpad}"""

prompt = PromptTemplate(
    template=react_template,
    input_variables=[
        "tools",
        "tool_names",
        "input",
        "agent_scratchpad"
    ]
)

In [29]:
from langchain_classic.agents import load_tools, Tool
from langchain_community.tools import DuckDuckGoSearchRun

search = DuckDuckGoSearchRun()

search_tool = Tool(
    name="duckduck",
    description="A web search engine. Use this as a search engine for general queries.",
    func=search.run,
)

tools = load_tools(
    ["llm-math"],
    llm=llm
)

tools.append(search_tool)

In [25]:
from langchain_classic.agents import load_tools

tools = load_tools(
    ["llm-math"],
    llm=openai_llm
)

tools.append(search_tool)

In [31]:
from langchain_classic.agents import AgentExecutor, create_react_agent

agent = create_react_agent(
    llm,
    tools,
    prompt
)

agent_executor = AgentExecutor(
    agent=agent,
    tools=tools,
    verbose=True,
    handle_parsing_errors=True
)

In [32]:
result = agent_executor.invoke({
    "input": "What is the current price of a MacBook Pro in USD? How much would it cost in EUR if the exchange rate is 0.85 EUR for 1 USD?"
})

print(result["output"])



> Entering new AgentExecutor chain...
 I need to perform a web search for the current price of MacBook Pro in USD and then calculate its cost in EUR using the given exchange rate.
Action: duckduck
Action Input: Current price of MacBook Pro in USDGet a new MacBook Pro laptop with M5, M5 Pro, or M5 Max from only $166.58 per month. Built for AI. Up to 24-hour battery life. Buy or lease now at apple.com.Students and educators — save on a new Mac. Get special pricing in the Education Store. Find MacBook laptops designed for speed and efficiency. Explore options with extended battery life and powerful performance. Compare prices and track the best Apple deals on MacBook, iMac and Mac mini models. Timestamps: 00:00 - MacBook Pro M4 Pro vs. Asus ProArt laptops comparison overview 00:29 - Build quality & usability insights for creators 00:57 - Form factor comparison: PX13, P16, and MacBook Pro 01:27 - Pen compatibility: PX13, P16 vs. MacBook Pro limitations 02:21... Pro Feature. Calibration M